# AdventureWorks — Statistical Analysis Notebook

Four analyses against the AdventureWorks PostgreSQL database:

1. **Sales Revenue Trend** — monthly OLS regression with residual diagnostics
2. **Customer Segmentation** — K-Means with elbow method and silhouette scoring
3. **Product Pricing** — list price vs. standard cost regression by category with markup distribution
4. **Purchasing & Vendor Analysis** — vendor spend clustering and monthly PO trend

In [1]:
# ---------------------------------------------------------------------------
# SETUP
# ---------------------------------------------------------------------------
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless backend for CI
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sqlalchemy import create_engine, text

# Seaborn theme
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

# Connection -- reads env vars when available (CI), falls back to local defaults
DB_HOST     = os.getenv('DB_HOST',     'localhost')
DB_PORT     = int(os.getenv('DB_PORT', 5432))
DB_NAME     = os.getenv('DB_NAME',     'adventureworks')
DB_USER     = os.getenv('DB_USER',     'awuser')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'Gunner!!24')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print(f'Connected to {DB_NAME} on {DB_HOST}:{DB_PORT}')

def sql(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

Connected to adventureworks on localhost:5432


---
## 1. Sales Revenue Trend & OLS Regression

In [2]:
sales = sql("""
    SELECT
        EXTRACT(YEAR  FROM orderdate)::int   AS yr,
        EXTRACT(MONTH FROM orderdate)::int   AS mo,
        TO_CHAR(orderdate, 'YYYY-MM')        AS year_month,
        COUNT(*)                             AS order_count,
        SUM(totaldue)                        AS total_revenue,
        AVG(totaldue)                        AS avg_order_value,
        COUNT(DISTINCT customerid)           AS unique_customers
    FROM sales.salesorderheader
    GROUP BY yr, mo, year_month
    ORDER BY year_month
""")

sales['total_revenue']   = sales['total_revenue'].astype(float).round(2)
sales['avg_order_value'] = sales['avg_order_value'].astype(float).round(2)

x = np.arange(len(sales))
y = sales['total_revenue'].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_sq      = round(r_value ** 2, 4)
ci_slope  = 1.96 * std_err          # 95% CI half-width on slope
trend     = slope * x + intercept
residuals = y - trend

print(f'OLS slope   : ${slope:,.0f} per month   (95% CI: +/- ${ci_slope:,.0f})')
print(f'R-squared   : {r_sq}')
print(f'P-value     : {p_value:.6f}')
print(f'Intercept   : ${intercept:,.0f}')

OLS slope   : $67,524 per month   (95% CI: +/- $36,135)
R-squared   : 0.2715
P-value     : 0.000797
Intercept   : $1,993,355


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Left: revenue bars + OLS line ---
ax = axes[0]
bar_colors = sns.color_palette('Blues_d', len(sales))
ax.bar(sales['year_month'], sales['total_revenue'], color=bar_colors, alpha=0.85, label='Monthly Revenue')
ax.plot(sales['year_month'], trend, color='crimson', linewidth=2.5,
        label=f'OLS Trend  (R\u00b2={r_sq},  p={p_value:.4f})')
ax.fill_between(
    sales['year_month'],
    trend - 1.96 * residuals.std(),
    trend + 1.96 * residuals.std(),
    alpha=0.12, color='crimson', label='95% prediction band'
)
ax.set_title('Monthly Sales Revenue with OLS Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
step = max(1, len(sales) // 10)
ax.set_xticks(sales['year_month'].iloc[::step])
ax.set_xticklabels(sales['year_month'].iloc[::step], rotation=45, ha='right')
ax.legend()

# --- Right: residuals ---
ax2 = axes[1]
ax2.bar(sales['year_month'], residuals,
        color=['#e05454' if r < 0 else '#4C72B0' for r in residuals], alpha=0.8)
ax2.axhline(0, color='black', linewidth=1)
ax2.set_title('OLS Residuals (Actual - Trend)')
ax2.set_xlabel('Month')
ax2.set_ylabel('Residual ($)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax2.set_xticks(sales['year_month'].iloc[::step])
ax2.set_xticklabels(sales['year_month'].iloc[::step], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('sales_trend_chart.png', bbox_inches='tight')
plt.show()
print('Saved: sales_trend_chart.png')

Saved: sales_trend_chart.png


In [4]:
# Summary stats table
summary = sales.groupby('yr').agg(
    months       = ('year_month', 'count'),
    total_rev    = ('total_revenue', 'sum'),
    avg_monthly  = ('total_revenue', 'mean'),
    orders       = ('order_count', 'sum'),
    customers    = ('unique_customers', 'sum')
).round(2).reset_index()
summary.columns = ['Year', 'Months', 'Total Revenue', 'Avg Monthly Revenue', 'Total Orders', 'Total Customers']
summary['Total Revenue']       = summary['Total Revenue'].map('${:,.0f}'.format)
summary['Avg Monthly Revenue'] = summary['Avg Monthly Revenue'].map('${:,.0f}'.format)
summary

,Year,Months,Total Revenue,Avg Monthly Revenue,Total Orders,Total Customers
0,2022,8,"$16,316,694","$2,039,587",1692,1692
1,2023,12,"$35,514,706","$2,959,559",3830,3830
2,2024,12,"$49,020,487","$4,085,041",14244,13849
3,2025,6,"$22,364,900","$3,727,483",11699,11373


---
## 2. Customer Segmentation (K-Means)

In [5]:
customers = sql("""
    SELECT
        customerid,
        COUNT(*)               AS order_count,
        SUM(totaldue)          AS total_spend,
        AVG(totaldue)          AS avg_order_value,
        MAX(totaldue)          AS max_order,
        MIN(orderdate)::date   AS first_order,
        MAX(orderdate)::date   AS last_order
    FROM sales.salesorderheader
    GROUP BY customerid
""")

customers['total_spend']     = customers['total_spend'].astype(float)
customers['avg_order_value'] = customers['avg_order_value'].astype(float)
customers['max_order']       = customers['max_order'].astype(float)

print(f'Customers: {len(customers):,}')
customers[['order_count','total_spend','avg_order_value']].describe().round(2)

Customers: 19,119


,order_count,total_spend,avg_order_value
count,19119.00,19119.00,19119.00
mean,1.65,6444.73,1678.12
std,1.46,43756.28,6034.78
min,1.00,1.52,1.52
25%,1.00,60.75,45.83
50%,1.00,606.62,606.62
75%,2.00,3119.15,2152.83
max,28.00,989184.08,151704.90


In [6]:
# Elbow method + silhouette scores to validate k
features = customers[['order_count', 'total_spend', 'avg_order_value']].fillna(0)
scaled   = StandardScaler().fit_transform(features)

inertias    = []
silhouettes = []
k_range     = range(2, 9)

for k in k_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(scaled, lbl))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='#4C72B0', linewidth=2)
axes[0].set_title('Elbow Method — Inertia by K')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].axvline(3, color='crimson', linestyle='--', linewidth=1.5, label='K=3 selected')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 's-', color='#55A868', linewidth=2)
axes[1].set_title('Silhouette Score by K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(3, color='crimson', linestyle='--', linewidth=1.5, label='K=3 selected')
axes[1].legend()

plt.tight_layout()
plt.savefig('customer_elbow_chart.png', bbox_inches='tight')
plt.show()

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f'Best silhouette score at K={best_k}  ({max(silhouettes):.4f})')

Best silhouette score at K=2  (0.9380)


In [7]:
# Fit final K=3 model
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
customers['cluster'] = km3.fit_predict(scaled)

# Label clusters by ascending mean spend
order_map = customers.groupby('cluster')['total_spend'].mean().sort_values().index
label_map = {c: l for c, l in zip(order_map, ['Low Value', 'Mid Value', 'High Value'])}
customers['segment'] = customers['cluster'].map(label_map)

palette = {'Low Value': '#4C72B0', 'Mid Value': '#DD8452', 'High Value': '#55A868'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: orders vs log10 spend
for seg, grp in customers.groupby('segment'):
    axes[0].scatter(
        grp['order_count'],
        np.log10(grp['total_spend'] + 1),
        label=seg, alpha=0.45, s=18, color=palette[seg]
    )
axes[0].set_title('Customer Segments: Orders vs Log10(Spend)')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('log10(Total Spend)')
axes[0].legend()

# Box: avg order value by segment
seg_order = ['Low Value', 'Mid Value', 'High Value']
sns.boxplot(
    data=customers,
    x='segment', y='avg_order_value',
    order=seg_order,
    palette=palette,
    ax=axes[1]
)
axes[1].set_title('Avg Order Value Distribution by Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Avg Order Value ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

plt.tight_layout()
plt.savefig('customer_segments_chart.png', bbox_inches='tight')
plt.show()

In [8]:
seg_summary = (
    customers.groupby('segment')
    .agg(
        customers    = ('customerid',     'count'),
        avg_orders   = ('order_count',    'mean'),
        avg_spend    = ('total_spend',    'mean'),
        median_spend = ('total_spend',    'median'),
        avg_order_val= ('avg_order_value','mean'),
        avg_max_order= ('max_order',      'mean'),
    )
    .round(2)
    .reindex(seg_order)
    .reset_index()
)
for col in ['avg_spend','median_spend','avg_order_val','avg_max_order']:
    seg_summary[col] = seg_summary[col].map('${:,.2f}'.format)
seg_summary

,segment,customers,avg_orders,avg_spend,median_spend,avg_order_val,avg_max_order
0,Low Value,18663,1.48,"$1,859.41",$463.51,"$1,030.98","$1,216.94"
1,Mid Value,304,8.72,"$74,873.75","$62,289.84","$12,813.74","$21,976.64"
2,High Value,152,7.76,"$432,585.76","$385,600.61","$58,864.84","$87,375.80"


---
## 3. Product Pricing Analysis — List Price vs. Standard Cost

In [9]:
products = sql("""
    SELECT
        p.name                          AS product_name,
        p.listprice                     AS list_price,
        p.standardcost                  AS standard_cost,
        p.listprice - p.standardcost    AS gross_margin,
        CASE WHEN p.standardcost > 0
             THEN ROUND(((p.listprice - p.standardcost) / p.standardcost * 100)::numeric, 1)
             ELSE NULL END              AS markup_pct,
        pc.name                         AS category,
        psc.name                        AS subcategory
    FROM production.product              p
    JOIN production.productsubcategory   psc ON psc.productsubcategoryid = p.productsubcategoryid
    JOIN production.productcategory      pc  ON pc.productcategoryid     = psc.productcategoryid
    WHERE p.listprice   > 0
      AND p.standardcost > 0
""")

products['list_price']    = products['list_price'].astype(float)
products['standard_cost'] = products['standard_cost'].astype(float)
products['gross_margin']  = products['gross_margin'].astype(float)
products['markup_pct']    = products['markup_pct'].astype(float)

print(f'Products: {len(products):,}')
products[['list_price','standard_cost','gross_margin','markup_pct']].describe().round(2)

Products: 295


,list_price,standard_cost,gross_margin,markup_pct
count,295.00,295.00,295.00,295.00
mean,744.60,438.22,306.37,95.58
std,892.56,534.90,361.96,39.22
min,2.29,0.86,1.43,29.90
25%,66.74,35.96,41.78,64.80
50%,337.22,199.85,162.94,82.60
75%,1100.24,650.42,407.41,125.20
max,3578.27,2171.29,1487.84,179.70


In [10]:
x2 = products['standard_cost'].values
y2 = products['list_price'].values

slope2, intercept2, r2, p2, se2 = stats.linregress(x2, y2)
r_sq2     = round(r2 ** 2, 4)
residuals2 = y2 - (slope2 * x2 + intercept2)

print(f'OLS: ListPrice = {slope2:.4f} * StandardCost + {intercept2:.2f}')
print(f'R-squared  : {r_sq2}')
print(f'P-value    : {p2:.6f}')
print(f'Std error  : {se2:.4f}')

x_line = np.linspace(x2.min(), x2.max(), 300)
y_line = slope2 * x_line + intercept2

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Scatter by category
cats    = products['category'].unique()
palette2 = sns.color_palette('tab10', len(cats))
for cat, col in zip(cats, palette2):
    sub = products[products['category'] == cat]
    axes[0].scatter(sub['standard_cost'], sub['list_price'],
                    label=cat, alpha=0.75, s=35, color=col)
axes[0].plot(x_line, y_line, color='crimson', linewidth=2.5, label=f'OLS (R\u00b2={r_sq2})')
axes[0].set_title('List Price vs Standard Cost by Category')
axes[0].set_xlabel('Standard Cost ($)')
axes[0].set_ylabel('List Price ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[0].legend(fontsize=8)

# Residuals vs fitted
fitted = slope2 * x2 + intercept2
axes[1].scatter(fitted, residuals2, alpha=0.5, s=25, color='#4C72B0')
axes[1].axhline(0, color='crimson', linewidth=1.5)
axes[1].set_title('Residuals vs Fitted Values')
axes[1].set_xlabel('Fitted List Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# Markup % distribution by category
sns.boxplot(
    data=products, x='category', y='markup_pct',
    palette='deep', ax=axes[2]
)
axes[2].set_title('Markup % Distribution by Category')
axes[2].set_xlabel('Category')
axes[2].set_ylabel('Markup (%)')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('product_pricing_chart.png', bbox_inches='tight')
plt.show()

OLS: ListPrice = 1.6633 * StandardCost + 15.72
R-squared  : 0.9935
P-value    : 0.000000
Std error  : 0.0078


In [11]:
pricing_summary = (
    products.groupby('category')
    .agg(
        products     = ('product_name',  'count'),
        avg_cost     = ('standard_cost', 'mean'),
        avg_price    = ('list_price',    'mean'),
        avg_margin   = ('gross_margin',  'mean'),
        avg_markup   = ('markup_pct',    'mean'),
        median_markup= ('markup_pct',    'median'),
    )
    .round(2)
    .reset_index()
)
for col in ['avg_cost','avg_price','avg_margin']:
    pricing_summary[col] = pricing_summary[col].map('${:,.2f}'.format)
for col in ['avg_markup','median_markup']:
    pricing_summary[col] = pricing_summary[col].map('{:.1f}%'.format)
pricing_summary

,category,products,avg_cost,avg_price,avg_margin,avg_markup,median_markup
0,Accessories,29,$13.23,$34.35,$21.12,161.4%,167.4%
1,Bikes,97,$949.41,"$1,586.74",$637.33,67.4%,60.9%
2,Clothing,35,$24.80,$50.99,$26.19,123.5%,142.4%
3,Components,134,$268.14,$469.86,$201.72,94.4%,82.6%


---
## 4. Purchasing & Vendor Analysis

In [12]:
vendors = sql("""
    SELECT
        v.name                                                     AS vendor_name,
        v.creditrating,
        COUNT(poh.purchaseorderid)                                 AS order_count,
        SUM(poh.subtotal + poh.taxamt + poh.freight)               AS total_spend,
        AVG(poh.subtotal + poh.taxamt + poh.freight)               AS avg_order_value,
        AVG(poh.freight)                                           AS avg_freight,
        SUM(poh.subtotal)                                          AS total_subtotal
    FROM purchasing.purchaseorderheader    poh
    JOIN purchasing.vendor                 v   ON v.businessentityid = poh.vendorid
    GROUP BY v.name, v.creditrating
    ORDER BY total_spend DESC
""")

vendors['total_spend']    = vendors['total_spend'].astype(float)
vendors['avg_order_value']= vendors['avg_order_value'].astype(float)

print(f'Vendors: {len(vendors):,}')
vendors.head(10)

Vendors: 86


,vendor_name,creditrating,order_count,total_spend,avg_order_value,avg_freight,total_subtotal
0,Superior Bicycles,1,50,5.034267e+06,100685.334800,2277.948800,4555897.500
1,Professional Athletic Consultants,1,50,3.379946e+06,67598.926430,1529.387510,3058774.950
2,Chicago City Saddles,1,51,3.347165e+06,65630.690127,1484.857245,3029108.775
3,Jackson Authority,1,51,2.821334e+06,55320.265020,1251.589725,2553243.000
4,"Vision Cycles, Inc.",1,50,2.777685e+06,55553.698220,1256.871020,2513742.000
5,Sport Fan Co.,1,50,2.675889e+06,53517.784320,1210.809600,2421619.200
6,"Proseware, Inc.",4,51,2.593901e+06,50860.810018,1150.697076,2347422.000
7,Crowley Sport,1,51,2.472770e+06,48485.687300,1096.961300,2237800.950
8,Greenwood Athletic Company,1,51,2.472770e+06,48485.687300,1096.961300,2237800.950
9,Mitchell Sports,1,50,2.424284e+06,48485.687300,1096.961300,2193922.500


In [13]:
# K-Means on vendors
vf      = vendors[['order_count','total_spend','avg_order_value']].fillna(0)
vs      = StandardScaler().fit_transform(vf)

v_inertias    = []
v_silhouettes = []
for k in range(2, 8):
    vkm  = KMeans(n_clusters=k, random_state=42, n_init=10)
    vlbl = vkm.fit_predict(vs)
    v_inertias.append(vkm.inertia_)
    v_silhouettes.append(silhouette_score(vs, vlbl))

# Fit K=3
vkm3 = KMeans(n_clusters=3, random_state=42, n_init=10)
vendors['cluster'] = vkm3.fit_predict(vs)

v_order = vendors.groupby('cluster')['total_spend'].mean().sort_values().index
v_labels= {c: l for c, l in zip(v_order, ['Low Spend','Mid Spend','High Spend'])}
vendors['segment'] = vendors['cluster'].map(v_labels)

v_palette = {'Low Spend':'#4C72B0','Mid Spend':'#DD8452','High Spend':'#55A868'}
v_seg_order = ['Low Spend','Mid Spend','High Spend']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter
for seg, grp in vendors.groupby('segment'):
    axes[0].scatter(
        grp['order_count'],
        np.log10(grp['total_spend'] + 1),
        label=seg, alpha=0.75, s=50, color=v_palette[seg]
    )
axes[0].set_title('Vendor Segments: Orders vs Log10(Spend)')
axes[0].set_xlabel('Purchase Order Count')
axes[0].set_ylabel('log10(Total Spend)')
axes[0].legend()

# Box: avg order value by segment
sns.boxplot(
    data=vendors, x='segment', y='avg_order_value',
    order=v_seg_order, palette=v_palette, ax=axes[1]
)
axes[1].set_title('Avg Order Value by Vendor Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Avg Order Value ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

plt.tight_layout()
plt.savefig('vendor_segments_chart.png', bbox_inches='tight')
plt.show()

In [14]:
# Pareto: top vendors by cumulative spend
vendors_sorted = vendors.sort_values('total_spend', ascending=False).reset_index(drop=True)
vendors_sorted['cumulative_pct'] = vendors_sorted['total_spend'].cumsum() / vendors_sorted['total_spend'].sum() * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(vendors_sorted.index, vendors_sorted['total_spend'], color='#4C72B0', alpha=0.7, label='Vendor Spend')
ax2 = ax.twinx()
ax2.plot(vendors_sorted.index, vendors_sorted['cumulative_pct'],
         color='crimson', linewidth=2, label='Cumulative %')
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, label='80% threshold')
ax.set_title('Vendor Spend — Pareto Distribution')
ax.set_xlabel('Vendor (ranked by spend)')
ax.set_ylabel('Total Spend ($)')
ax2.set_ylabel('Cumulative Spend (%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.savefig('vendor_pareto_chart.png', bbox_inches='tight')
plt.show()

# How many vendors make up 80% of spend?
n80 = (vendors_sorted['cumulative_pct'] <= 80).sum() + 1
pct80 = round(n80 / len(vendors_sorted) * 100, 1)
print(f'{n80} vendors ({pct80}% of all vendors) account for 80% of total spend')

25 vendors (29.1% of all vendors) account for 80% of total spend


In [15]:
# Monthly PO trend
po_trend = sql("""
    SELECT
        TO_CHAR(orderdate, 'YYYY-MM')               AS year_month,
        COUNT(*)                                     AS order_count,
        SUM(subtotal + taxamt + freight)             AS total_spend,
        AVG(subtotal + taxamt + freight)             AS avg_order_value
    FROM purchasing.purchaseorderheader
    GROUP BY year_month
    ORDER BY year_month
""")
po_trend['total_spend']     = po_trend['total_spend'].astype(float)
po_trend['avg_order_value'] = po_trend['avg_order_value'].astype(float)

# OLS on PO spend
xp  = np.arange(len(po_trend))
yp  = po_trend['total_spend'].values
sp, ip, rp, pp, sep = stats.linregress(xp, yp)
r_sqp = round(rp ** 2, 4)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

step = max(1, len(po_trend) // 10)

axes[0].bar(po_trend['year_month'], po_trend['total_spend'],
            color=sns.color_palette('Greens_d', len(po_trend)), alpha=0.85)
axes[0].plot(po_trend['year_month'], sp * xp + ip,
             color='crimson', linewidth=2, label=f'OLS Trend (R\u00b2={r_sqp})')
axes[0].set_title('Monthly Purchasing Spend with Trend')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Total Spend ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
axes[0].set_xticks(po_trend['year_month'].iloc[::step])
axes[0].set_xticklabels(po_trend['year_month'].iloc[::step], rotation=45, ha='right')
axes[0].legend()

axes[1].bar(po_trend['year_month'], po_trend['order_count'],
            color=sns.color_palette('Blues_d', len(po_trend)), alpha=0.85)
axes[1].set_title('Monthly Purchase Order Count')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Order Count')
axes[1].set_xticks(po_trend['year_month'].iloc[::step])
axes[1].set_xticklabels(po_trend['year_month'].iloc[::step], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('po_trend_chart.png', bbox_inches='tight')
plt.show()

print(f'PO Spend OLS: ${sp:,.0f}/month trend,  R2={r_sqp},  p={pp:.4f}')

PO Spend OLS: $193,524/month trend,  R2=0.4532,  p=0.0000


In [16]:
vendor_summary = (
    vendors.groupby('segment')
    .agg(
        vendors       = ('vendor_name',   'count'),
        avg_orders    = ('order_count',   'mean'),
        avg_spend     = ('total_spend',   'mean'),
        total_spend   = ('total_spend',   'sum'),
        avg_order_val = ('avg_order_value','mean'),
    )
    .round(2)
    .reindex(v_seg_order)
    .reset_index()
)
for col in ['avg_spend','total_spend','avg_order_val']:
    vendor_summary[col] = vendor_summary[col].map('${:,.2f}'.format)
vendor_summary

,segment,vendors,avg_orders,avg_spend,total_spend,avg_order_val
0,Low Spend,53,50.64,"$184,984.69","$9,804,188.68","$3,641.72"
1,Mid Spend,7,1.71,"$340,110.40","$2,380,772.81","$204,448.98"
2,High Spend,26,50.62,"$2,242,091.20","$58,294,371.15","$44,347.84"


---
*Generated by analysis.ipynb — AdventureWorks PostgreSQL*